In [ ]:
from concurrent.futures import ProcessPoolExecutor
import itertools
import os
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np
from astropy import units as u
from astropy.visualization import quantity_support
from tqdm import tqdm
quantity_support()
import scipy as sp
from scipy.stats._stats_py import Power_divergenceResult

from spectrum_component_analyser.internals.readers.JWST.instruments import NIRISS, NIRSPEC
from spectrum_component_analyser.interpolated_spectrum import get_interpolated_phoenix_spectrum
from spectrum_component_analyser.internals.phoenix_spectrum import phoenix_spectrum
from spectrum_component_analyser.internals.readers.JWST.file_reader import JWSTFileReader
from spectrum_component_analyser.internals.readers.JWST.folder_reader import JWSTFolderReader, JWSTTargets
from spectrum_component_analyser.internals.spectral_grid import download_spectrum, get_wavelength_grid, spectral_grid
from spectrum_component_analyser.internals.spectrum import spectrum

target : JWSTTargets = JWSTTargets.LTT3780

all_spectra : list[spectrum] = JWSTFolderReader.get_all_spectra(target=target, instrument=NIRISS)

print(all_spectra)

example_spectrum : spectrum = all_spectra[0]

mask = np.isfinite(example_spectrum.Fluxes)

example_spectrum = example_spectrum[mask]

# normalisation = np.sum(jwst_spectrum.Fluxes)
# jwst_spectrum.Fluxes /= normalisation

spec_grid : spectral_grid = spectral_grid.from_hdf5(grid_name="JWST_convolved_not_oversmoothed.hdf5")

In [ ]:
"""
find the standard deviation of each point
find an averaged spectrum
analyse that average spectrum using a (now correct) Pearson chi-squared test
"""

min_included_integration_index = 0
max_included_integration_index = 10

all_fluxes=[spec.Fluxes for spec in all_spectra[min_included_integration_index:max_included_integration_index]]

average_spectrum = spectrum(wavelengths=all_spectra[0].Wavelengths,
                                 fluxes=np.mean(all_fluxes, axis=0) * all_fluxes[0].unit,
                                 normalised_point=None,
                                 temperature = None,
                                 observational_resolution=None,
                                 observational_wavelengths=None,
                                 name="averaged" + target.value)

average_spectrum = average_spectrum[mask]

flux_std_deviation = np.std(all_fluxes, axis=0) * all_fluxes[0].unit

flux_std_deviation = flux_std_deviation[mask]

fig, ax = plt.subplots(figsize=(26,10))
average_spectrum.plot()
plt.errorbar(average_spectrum.Wavelengths, average_spectrum.Fluxes, yerr=flux_std_deviation, ecolor='red', capsize=2)
plt.scatter(average_spectrum.Wavelengths, flux_std_deviation, s=1)
plt.show()


In [ ]:
# sp.optimize.shgo(
#     func=compute_error,
#     bounds=[(np.min(spec_grid.T_effs.value), np.max(spec_grid.T_effs.value)), (np.min(spec_grid.FeHs.value), np.max(spec_grid.FeHs.value)), (np.min(spec_grid.Log_gs.value), np.max(spec_grid.Log_gs.value))],
#     callback=print
# )
# %matplotlib inline
ts = np.linspace(np.min(spec_grid.T_effs), np.max(spec_grid.T_effs), 1000)
fehs = np.linspace(np.min(spec_grid.FeHs), np.max(spec_grid.FeHs), 20)
loggs = np.linspace(np.min(spec_grid.Log_gs), np.max(spec_grid.Log_gs), 5)
# loggs = [ 4.85 * u.dimensionless_unscaled ]

# chi2_grid = np.zeros((len(ts), len(fehs), len(loggs)))

params = list(itertools.product(ts, fehs, loggs))

# create "dynamical mask" as per https://www.aanda.org/articles/aa/pdf/2016/03/aa22261-13.pdf

total_fluxes : np.ndarray = np.zeros((len(spec_grid.Wavelengths))) * u.Jy

all_phoenix_params = list(itertools.product(spec_grid.T_effs, spec_grid.FeHs, spec_grid.Log_gs))

for t, f, l in all_phoenix_params:
    total_fluxes += spec_grid.LookupTable[t, f, l]

average_flux = total_fluxes / len(all_phoenix_params)

average_flux = average_flux[mask]

def compute_error(params):
    t, f, l = params
    t *= u.K
    f *= u.dimensionless_unscaled
    l *= u.dimensionless_unscaled

    spec : phoenix_spectrum = get_interpolated_phoenix_spectrum(t, f, l, star_name=target.value, spec_grid=spec_grid)

    spec = spec[mask]

    # print(average_flux)
    dynamical_mask = spec.Fluxes / average_flux # element-wise division

    # plt.plot(spec_grid.Wavelengths[mask], dynamical_mask)
    # plt.show()

    # normalise for chi-squared
    # spec.Fluxes /= normalisation

    # spec.plot(clear=True)
    # jwst_spectrum.plot(clear=False)
    # plt.legend()
    # plt.show()

    e = np.sum(((average_spectrum.Fluxes - spec.Fluxes)**2 / flux_std_deviation).value)

    return e



with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(compute_error, params), total=len(params)))

chi2_grid = np.array(results).reshape((len(ts), len(fehs), len(loggs)))
# chi2_grid *= u.Jy

In [ ]:
# test if the interpolation is actually working

get_interpolated_phoenix_spectrum(3500 * u.K, 0.0 * u.dimensionless_unscaled, 5.0 * u.dimensionless_unscaled, star_name="interp", spec_grid=spec_grid).plot()
f = spec_grid.LookupTable[3500 * u.K, 0.0 * u.dimensionless_unscaled, 5.0 * u.dimensionless_unscaled]

s = spectrum(spec_grid.Wavelengths, f, None, None, None, None, name="test")
# s.Fluxes *= 1.1
s.plot(clear=False)
plt.legend()
plt.show()

In [ ]:
# Label the best fit point
min_idx = np.unravel_index(np.argmin(chi2_grid), chi2_grid.shape)
optimal_t = ts[min_idx[0]]
optimal_feh = fehs[min_idx[1]]
optimal_logg = loggs[min_idx[2]]
min_chi2 = chi2_grid[min_idx]

%matplotlib inline
k_slice = min_idx[2]
selected_logg = optimal_logg

# Slice the 3D grid to get a 2D (Teff, FeH) array
plot_data = chi2_grid[:, :, k_slice]


plt.figure(figsize=(10, 7))

# Create the plot
# Shading='gouraud' smooths the grid; 'auto' keeps it pixelated
pcm = plt.pcolormesh(fehs, ts, plot_data, cmap='viridis_r', shading='auto')

# Add a colorbar to show the chi-squared magnitude
cbar = plt.colorbar(pcm)
cbar.set_label(r'$\chi^2$ (Residual Sum of Squares)', rotation=270, labelpad=15)


print(optimal_t, optimal_feh, optimal_logg)
print(min_chi2)

plt.scatter(optimal_feh, optimal_t, color='red', marker='x', s=100, label='Best Fit')

plt.xlabel('Metallicity [Fe/H]')
plt.ylabel('Effective Temperature $T_{eff}$ [K]')
plt.title(f'Chi-Squared Surface for log(g) = {loggs[0]}')
plt.legend()
plt.show()

average_spectrum.plot()
i = get_interpolated_phoenix_spectrum(optimal_t, optimal_feh, optimal_logg, star_name="interpolated", spec_grid=spec_grid)
# i = get_interpolated_phoenix_spectrum(3300 * u.K, .0 * u.dimensionless_unscaled, 4.85 * u.dimensionless_unscaled, star_name="test", spec_grid=spec_grid)
# i.Fluxes /= normalisation
i.plot(clear=False)
plt.legend()
plt.show()

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import numpy as np

# 1. Setup the figure and axis
fig, ax = plt.subplots(figsize=(10, 8))
plt.subplots_adjust(bottom=0.25) # Make room for the slider

# 2. Initial plot (default to the first logg)
k_init = 0
# Use np.log as you did in your snippet
pcm = ax.pcolormesh(fehs, ts, chi2_grid[:, :, k_init], 
                    cmap='viridis_r', shading='auto')

# 3. Add formatting and the 'Best Fit' marker
cbar = fig.colorbar(pcm, ax=ax)
cbar.set_label(r'$\ln(\chi^2)$', rotation=270, labelpad=15)

# Calculate global best fit once
min_idx = np.unravel_index(np.argmin(chi2_grid), chi2_grid.shape)
best_i, best_j, best_k = min_idx

# We only show the marker if it's on the current slice (optional logic)
fit_marker = ax.scatter(fehs[best_j], ts[best_i], color='red', 
                        marker='x', s=100, label='Global Best Fit')

ax.set_xlabel('Metallicity [Fe/H]')
ax.set_ylabel('Effective Temperature $T_{eff}$ [K]')
ax_title = ax.set_title(f'Chi-Squared Surface: log(g) = {loggs[k_init]:.2f}')
ax.legend()

# 4. Create the Slider axes [left, bottom, width, height]
ax_slider = plt.axes([0.2, 0.1, 0.6, 0.03])
slider = Slider(
    ax=ax_slider,
    label='log(g) ',
    valmin=0,
    valmax=len(loggs) - 1,
    valinit=k_init,
    valfmt='%d' # Display as integer index
)

# 5. The Update Function
def update(val):
    k = int(slider.val)
    # Update the data in the pcolormesh
    new_data = np.log(chi2_grid[:, :, k])
    pcm.set_array(new_data.ravel())
    
    # Update title and visibility of the best-fit marker
    ax_title.set_text(f'Chi-Squared Surface: log(g) = {loggs[k]:.2f}')
    
    # Optional: Highlight best fit only if on the correct slice
    fit_marker.set_visible(k == best_k)
    
    fig.canvas.draw_idle()

slider.on_changed(update)

plt.show()